# Crafting an AI-Powered HR Assistant: A Use Case for Nestle’s HR Policy Documents

## Overview

The project aims to create a conversational chatbot that responds to user inquiries using PDF document information. It requires proficiency in extracting and converting text into numerical vectors, establishing an answer-finding mechanism, and designing a user-friendly chatbot interface with Gradio. Additionally, the initiative emphasizes structuring inquiries for clear communication and deploying the chatbot for practical use, guaranteeing the system's accessibility and efficiency in meeting user needs.

## Instructions

Review the learning materials and the Gradio documentation provided for the project.
Read the sections on situation, task, action, and result carefully to understand the assignment.
Complete and submit the assignment through the Learning Management System (LMS).
Adhere closely to the provided guidelines, ensuring your submission contains all necessary analyses and interpretations.

## Situation

As a developer, you have received the critical task of improving the operational efficiency of Nestlé's human resources department, a leading multinational corporation. Your toolkit includes cutting-edge conversational AI technology, Python libraries, the powerful GPT model from OpenAI, and the user-friendly Gradio UI. Your mission is to integrate these advanced tools seamlessly to transform HR processes, creating a more streamlined and efficient workflow within the Nestlé organization.

# Task

Your task is to develop a conversational chatbot. This chatbot must answer queries about Nestlé's HR reports efficiently. Use Python libraries, OpenAI's GPT model, and Gradio UI. These tools will help you create a user-friendly interface. This interface will extract and process information from documents. It will provide accurate responses to user queries.

## Action

- Import essential tools and set up OpenAI's API environment.
- Load Nestle's HR policy using PyPDFLoader and split it for easy processing.
- Create vector representations for text chunks using Chroma dB and OpenAI's embeddings.
- Build a question-answering system using the GPT-3.5 Turbo model to retrieve answers from text chunks.
- Create a prompt template to guide the chatbot in understanding and responding to users.
- Use Gradio to build a user-friendly chatbot interface, enabling interaction and information retrieval.

## Result

Upon completing this project, you will submit an IPYNB file demonstrating your ability to use advanced AI and machine learning technologies to develop a conversational chatbot. Your submission must include the entire workflow: setting up the programming environment, processing text documents, creating text vector representations, and building a question-answering system. Ensure the interface is user-friendly to facilitate effective interaction and information retrieval.


## Resources
- Gradio documentation website: https://www.gradio.app/docs
- Gradio Quickstart: https://www.gradio.app/guides/quickstart
- Gradio PDF [Gradio_Documentation.pdf](./Gradio_Documentation.pdf)
- Nestle HR Policy [Nestle_HR_Policy.pdf](./Nestle_HR_Policy.pdf)

## Step 1: Install Required Dependencies

First, install all necessary packages. Run this cell if packages are not already installed.

In [13]:
# Install required packages (uncomment and run if needed)
# !pip install langchain langchain-openai langchain-community openai chromadb pypdf gradio python-dotenv tiktoken

## Step 2: Import Libraries and Set Up Environment

Import essential tools and configure the OpenAI API environment.

In [14]:
import os
import gradio as gr
from dotenv import load_dotenv

# LangChain imports
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# Load environment variables from .env file (if exists)
load_dotenv()

# Set up OpenAI API key
# Option 1: Set directly (not recommended for production)
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"

# Option 2: Load from environment variable (recommended)
# Make sure OPENAI_API_KEY is set in your environment or .env file

# Verify API key is set
if not os.getenv("OPENAI_API_KEY"):
    print("Warning: OPENAI_API_KEY not found. Please set it before running the chatbot.")
else:
    print("OpenAI API key loaded successfully.")

OpenAI API key loaded successfully.


## Step 3: Load and Process the Nestle HR Policy PDF

Load the PDF document using PyPDFLoader and split it into manageable text chunks for processing.

In [15]:
# Load the Nestle HR Policy PDF
pdf_path = "./the_nestle_hr_policy_2012.pdf"

# Initialize the PDF loader
loader = PyPDFLoader(pdf_path)

# Load the document
documents = loader.load()

print(f"Loaded {len(documents)} pages from the PDF.")
print(f"\nSample content from page 1:\n{documents[0].page_content[:500]}...")

Loaded 8 pages from the PDF.

Sample content from page 1:
Policy
MandatorySeptember   2012
The Nestlé  
Human Resources Policy...


In [16]:
# Split documents into smaller chunks for better processing
# Using RecursiveCharacterTextSplitter for optimal chunk boundaries

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # Maximum size of each chunk
    chunk_overlap=200,    # Overlap between chunks to maintain context
    length_function=len,
    separators=["\n\n", "\n", " ", ""]  # Priority order for splitting
)

# Split the documents into chunks
text_chunks = text_splitter.split_documents(documents)

print(f"Created {len(text_chunks)} text chunks from the documents.")
print(f"\nSample chunk:\n{text_chunks[0].page_content}")

Created 20 text chunks from the documents.

Sample chunk:
Policy
MandatorySeptember   2012
The Nestlé  
Human Resources Policy


## Step 4: Create Vector Store with ChromaDB and OpenAI Embeddings

Convert text chunks into vector representations using OpenAI embeddings and store them in ChromaDB for efficient similarity search.

In [17]:
# Initialize OpenAI embeddings
embeddings = OpenAIEmbeddings(
    model="text-embedding-ada-002"  # OpenAI's embedding model
)

# Create ChromaDB vector store from the text chunks
# This converts each chunk into a numerical vector representation
vectorstore = Chroma.from_documents(
    documents=text_chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"  # Persist the database locally
)

print(f"Vector store created with {vectorstore._collection.count()} vectors.")
print("Database persisted to ./chroma_db")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector store created with 40 vectors.
Database persisted to ./chroma_db


In [18]:
# Create a retriever from the vector store
# This will be used to find relevant document chunks for each query
retriever = vectorstore.as_retriever(
    search_type="similarity",  # Use similarity search
    search_kwargs={"k": 4}     # Return top 4 most relevant chunks
)

# Test the retriever with a sample query
test_query = "What is Nestle's hiring policy?"
relevant_docs = retriever.invoke(test_query)

print(f"Test Query: {test_query}")
print(f"\nFound {len(relevant_docs)} relevant documents.")
print(f"\nFirst relevant chunk:\n{relevant_docs[0].page_content[:300]}...")

Test Query: What is Nestle's hiring policy?

Found 4 relevant documents.

First relevant chunk:
Policy
MandatorySeptember   2012
The Nestlé  
Human Resources Policy...


## Step 5: Build Question-Answering System with GPT-3.5 Turbo

Create a RetrievalQA chain that combines the retriever with OpenAI's GPT-3.5 Turbo model. A custom prompt template guides the chatbot in understanding and responding to HR-related queries.

In [19]:
# Define a custom prompt template for the HR Assistant
# This guides the model to provide helpful, accurate responses based on the HR policy documents

prompt_template = """You are a helpful HR Assistant for Nestlé. Your role is to answer questions about 
Nestlé's Human Resources policies based on the official HR Policy document.

Use the following context from the Nestlé HR Policy document to answer the question. 
If you cannot find the answer in the provided context, say "I don't have enough information 
in the HR policy document to answer this question. Please contact your HR representative for more details."

Be professional, helpful, and concise in your responses. When relevant, cite specific policies 
or sections from the document.

Context:
{context}

Question: {question}

Answer:"""

# Create the prompt template
PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

print("Prompt template created successfully.")

Prompt template created successfully.


In [20]:
# Initialize the ChatOpenAI LLM with GPT-3.5 Turbo
llm = ChatOpenAI(
    model_name="gpt-3.5-turbo",
    temperature=0.3,  # Lower temperature for more consistent, factual responses
    max_tokens=500    # Limit response length
)

# Create the RetrievalQA chain
# This combines the retriever, LLM, and prompt into a single chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",  # "stuff" method puts all retrieved docs into the prompt
    retriever=retriever,
    return_source_documents=True,  # Return source documents for transparency
    chain_type_kwargs={"prompt": PROMPT}
)

print("QA Chain created successfully with GPT-3.5 Turbo.")

QA Chain created successfully with GPT-3.5 Turbo.


In [21]:
# Test the QA chain with a sample question
test_question = "What is Nestlé's policy on employee training and development?"

response = qa_chain.invoke({"query": test_question})

print(f"Question: {test_question}")
print(f"\nAnswer: {response['result']}")
print(f"\n--- Source Documents Used ---")
for i, doc in enumerate(response['source_documents'][:2], 1):
    print(f"\nSource {i}: {doc.page_content[:200]}...")

Question: What is Nestlé's policy on employee training and development?

Answer: Nestlé's policy on employee training and development emphasizes that learning is part of the company culture. Employees at all levels are encouraged to upgrade their knowledge and skills. The company determines training and development priorities, with responsibility for implementation shared between employees, line managers, and Human Resources. Experience and on-the-job training are primary sources of learning, and managers are responsible for guiding and coaching employees to succeed in their current positions. Continuous improvement, sharing knowledge, and participating in lateral professional development, extension of responsibilities, and cross-functional teams are also encouraged. Nestlé offers a comprehensive range of training activities and methodologies to support employee development.

--- Source Documents Used ---

Source 1: The Nestlé Human Resources Policy4
Learning is part of the Company cul

## Step 6: Create Gradio Chatbot Interface

Build a user-friendly chatbot interface using Gradio that allows users to interact with the HR Assistant and retrieve information from the Nestlé HR Policy document.

In [22]:
# Define the chatbot response function
def hr_assistant_chat(message, history):
    """
    Process user message and generate response using the QA chain.
    
    Args:
        message: User's question
        history: Chat history (list of [user_msg, assistant_msg] pairs)
    
    Returns:
        Assistant's response
    """
    try:
        # Get response from the QA chain
        response = qa_chain.invoke({"query": message})
        answer = response['result']
        
        # Optionally include source information
        sources = response.get('source_documents', [])
        if sources:
            answer += "\n\n---\n*Based on Nestlé HR Policy Document*"
        
        return answer
    
    except Exception as e:
        return f"I apologize, but I encountered an error processing your question. Please try again or rephrase your question. Error: {str(e)}"

In [23]:
# Create the Gradio ChatInterface
# This provides a modern, user-friendly chat interface

demo = gr.ChatInterface(
    fn=hr_assistant_chat,
    title="Nestlé HR Policy Assistant",
    description="""Welcome to the Nestlé HR Policy Assistant! 
    
I can answer questions about Nestlé's Human Resources policies, including:
- Hiring and recruitment policies
- Training and development opportunities
- Performance management
- Employee relations
- Working conditions and benefits
- Career development

Ask me anything about Nestlé's HR policies!""",
    examples=[
        "What is Nestlé's hiring policy?",
        "How does Nestlé handle employee training and development?",
        "What are the Total Rewards at Nestlé?",
        "What is Nestlé's policy on employee relations?",
        "How does Nestlé support work-life balance?",
        "What is the role of line managers at Nestlé?"
    ],
    theme=gr.themes.Soft(),
    retry_btn="Retry",
    undo_btn="Undo",
    clear_btn="Clear Chat"
)

print("Gradio interface created successfully!")

Gradio interface created successfully!


In [ ]:
# Launch the Gradio interface
# The interface will be accessible in the notebook output and via a local URL

demo.launch(
    share=False,        # Set to True to create a public shareable link
    inline=True,        # Display inline in the notebook
    debug=True          # Enable debug mode for troubleshooting
)

Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


## Summary

This AI-Powered HR Assistant demonstrates the following Generative AI best practices:

### Architecture
1. **RAG (Retrieval-Augmented Generation)**: Combines document retrieval with LLM generation for accurate, contextual responses
2. **Vector Embeddings**: Uses OpenAI's embeddings to convert text into numerical vectors for semantic search
3. **ChromaDB**: Persistent vector store for efficient similarity search across document chunks

### Best Practices Implemented
- **Text Chunking**: Documents split into optimal chunks (1000 chars) with overlap (200 chars) to maintain context
- **Custom Prompt Engineering**: Structured prompt template guides the LLM to provide professional, policy-based responses
- **Temperature Control**: Low temperature (0.3) for consistent, factual responses
- **Source Attribution**: Responses include source document references for transparency
- **Error Handling**: Graceful error handling in the chatbot function
- **User-Friendly Interface**: Gradio provides an intuitive chat interface with example questions

### Technologies Used
- **LangChain**: Orchestration framework for LLM applications
- **OpenAI GPT-3.5 Turbo**: Language model for generating responses
- **OpenAI Embeddings**: text-embedding-ada-002 for vector representations
- **ChromaDB**: Vector database for document storage and retrieval
- **Gradio**: Web UI framework for the chatbot interface
- **PyPDFLoader**: PDF document processing